In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

path = Path('catalog.dat.txt')

COLS = [
    ('ID', 0, 3),
    ('RAdeg', 4, 22),
    ('DEdeg', 23, 42),
    ('FWHM', 60, 65),
    ('FOIII', 138, 148),
    ('FHb', 102, 111),
    ('l_FHb', 101, 102),
    ('FHa', 211, 220),
    ('l_FHa', 210, 211),
    ('FNII', 249, 258),
    ('l_FNII', 248, 249),
    ('FSII', 287, 296),
    ('l_FSII', 286, 287),
    ('OIII_SNR', 167, 174),
    ('m5007', 323, 329),
    ('OIII_BKG', 157, 166),
    ('HA_BKG', 230, 240)]

def parse(x):
    x = x.strip()
    return float(x) if x not in ('', '---', '-----') else np.nan

def read_catalog(path):
    rows = []
    for line in path.read_text().splitlines():
        row = {}
        for name, start, end in COLS:
            raw = line[start:end]
            if name.startswith('l_'):
                row[name] = raw.strip()
            elif name == 'ID':
                row[name] = int(raw.strip())
            else:
                row[name] = parse(raw)
        rows.append(row)
    return pd.DataFrame(rows)

def add_ratios(df):
    df = df.copy()
    df['log_OIII_Hb'] = np.log10(df['FOIII'] / df['FHb'])
    df['log_NII_Ha'] = np.log10(df['FNII'] / df['FHa'])
    df['log_SII_Ha'] = np.log10(df['FSII'] / df['FHa'])
    return df

def apply_masks(df):
    base = (
        (df['FOIII'] > 0) & np.isfinite(df['FOIII']) &
        (df['FHb'] > 0) & np.isfinite(df['FHb']) & (df['l_FHb'] == '') &
        (df['FHa'] > 0) & np.isfinite(df['FHa']) & (df['l_FHa'] == ''))

    mask_nii = base & ((df['FNII'] > 0) & np.isfinite(df['FNII']) & (df['l_FNII'] == ''))

    mask_sii = base & ((df['FSII'] > 0) & np.isfinite(df['FSII']) & (df['l_FSII'] == ''))

    return mask_nii, mask_sii


df = read_catalog(path)
print(df.columns)
df = add_ratios(df)

mask_nii, mask_sii = apply_masks(df)

out = df[['ID', 'RAdeg', 'DEdeg', 'FWHM', 'log_OIII_Hb', 'log_NII_Ha', 'log_SII_Ha', 'm5007','OIII_SNR', 'OIII_BKG', 'HA_BKG']].copy()
out['use_NII'] = mask_nii
out['use_SII'] = mask_sii
print(f"NII valid: {mask_nii.sum()}")
print(f"SII valid: {mask_sii.sum()}")

Index(['ID', 'RAdeg', 'DEdeg', 'FWHM', 'FOIII', 'FHb', 'l_FHb', 'FHa', 'l_FHa',
       'FNII', 'l_FNII', 'FSII', 'l_FSII', 'OIII_SNR', 'm5007', 'OIII_BKG',
       'HA_BKG'],
      dtype='str')
NII valid: 174
SII valid: 50


In [13]:
out.to_csv('PN_CAT.csv', index=False)